In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
results_df = spark.table(bronze_table)
display(results_df)

In [0]:
from pyspark.sql import functions as F

In [0]:
results_dropped_df = results_df.drop(F.col("url"))

In [0]:
results_renamed_df = (
    results_dropped_df
        .withColumnsRenamed({
            "constructorId": "constructor_id"
            , "driverId": "driver_id"
            , "raceName": "race_name"
            , "date": "race_date"
            , "grid": "grid_position"
            , "laps": "completed_laps"
            , "number": "car_number"
            , "position": "final_position"
            , "positionText": "final_position_text"
        })
)
display(results_renamed_df)

In [0]:
results_valid_df = (
    results_renamed_df
        .filter(
            # F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("constructor_id").isNotNull() &
            F.col("driver_id").isNotNull()
        )
)
display(results_valid_df)

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season", "round", "constructor_id", "driver_id"])

In [0]:
display(results_renamed_df.count() - results_distinct_df.count())

In [0]:
results_final_df = (
    results_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

In [0]:
write_to_silver(
    input_df=results_final_df,
    target_table=silver_table,
    merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update=[
        "race_name",
        "race_date",
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))